# Week 8 – Text Classification: Trump Tweet Sentiment

**Pipeline:** Raw Tweet → Clean Text → TF-IDF → Logistic Regression → Evaluate

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Helper Function for Text Cleaning

Bring in the same helper functions from the preprocessing notebook.

In [2]:
import pandas as pd
import re
import warnings
warnings.filterwarnings('ignore')

import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

stop_words = set(stopwords.words('english'))
stop_words.update(['@', 'RT'])
print('All imports ready.')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


All imports ready.


[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


In [3]:
def lower_order(text):
    """Convert text to lowercase."""
    return text.lower()

def remove_urls(text):
    """Remove http/https/www links."""
    return re.compile(r'https?://\S+|www\.\S+').sub('', text)

def remove_emoji(string):
    """Replace emoji characters with a space."""
    emoji_pattern = re.compile(
        '['
        u'\U0001F600-\U0001F64F'
        u'\U0001F300-\U0001F5FF'
        u'\U0001F680-\U0001F6FF'
        u'\U0001F1E0-\U0001F1FF'
        u'\U00002702-\U000027B0'
        u'\U000024C2-\U0001F251'
        ']+', flags=re.UNICODE
    )
    return emoji_pattern.sub(' ', string)

def removeunwanted_characters(document):
    """Remove @mentions, #hashtags, punctuation, emojis, and extra spaces."""
    document = re.sub(r'@[A-Za-z0-9_]+', ' ', document)
    document = re.sub(r'#[A-Za-z0-9_]+', '', document)
    document = re.sub(r'[^0-9A-Za-z ]', '', document)
    document = remove_emoji(document)
    document = document.replace('  ', ' ')
    return document.strip()

print('Helper functions defined.')

Helper functions defined.


## Build Text Cleaning Pipeline

In [4]:
wordnet = WordNetLemmatizer()
porter  = PorterStemmer()

def text_cleaning_pipeline(dataset, rule='lemmatize'):
    """Lowercase → remove URLs/emojis/noise → tokenise → stopwords → lemmatise or stem."""
    data = lower_order(dataset)
    data = remove_urls(data)
    data = remove_emoji(data)
    data = removeunwanted_characters(data)
    tokens = data.split()
    tokens = [w for w in tokens if w not in stop_words]
    if rule == 'lemmatize':
        tokens = [wordnet.lemmatize(w, pos='v') for w in tokens]
    elif rule == 'stem':
        tokens = [porter.stem(w) for w in tokens]
    else:
        print("Pick 'lemmatize' or 'stem'")
    return ' '.join(tokens)

# Quick sanity check
print(text_cleaning_pipeline("RT @CNN: Trump visits https://t.co/abc #Trump 👋"))

rt trump visit


## Step 1 – Load the Dataset

In [5]:
data = pd.read_csv('/content/drive/MyDrive/workshop8/trum_tweet_sentiment_analysis.csv', encoding='ISO-8859-1')
print('Full dataset shape:', data.shape)
print('Label counts:\n', data['Sentiment'].value_counts())
data.head()

Full dataset shape: (1850123, 2)
Label counts:
 Sentiment
0    1244211
1     605912
Name: count, dtype: int64


,text,Sentiment
0,RT @JohnLeguizamo: #trump not draining swamp b...,0
1,ICYMI: Hackers Rig FM Radio Stations To Play A...,0
2,Trump protests: LGBTQ rally in New York https:...,1
3,"""Hi I'm Piers Morgan. David Beckham is awful b...",0
4,RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...,0


In [6]:
# Use a 50 000-row sample so training stays fast
data = data.dropna(subset=['text', 'Sentiment'])
data = data.sample(n=50000, random_state=42).reset_index(drop=True)
print('Working size:', data.shape)
print(data['Sentiment'].value_counts())

Working size: (50000, 2)
Sentiment
0    33620
1    16380
Name: count, dtype: int64


## Step 2 – Text Cleaning & Tokenisation

In [7]:
print('Cleaning tweets … (this may take ~1 min)')
data['cleaned_text'] = data['text'].apply(lambda t: text_cleaning_pipeline(t))
print('Done!')
print('\nBefore:', data['text'].iloc[0])
print('After :', data['cleaned_text'].iloc[0])

Cleaning tweets … (this may take ~1 min)
Done!

Before: RT @EthandeMarsi: I don't understand why adults would be mean to a kid! We're the same age! @realDonaldTrump , I'll be friends w/BarrÂ https://t.co/1zFUGhrU0s
After : rt dont understand adults would mean kid age ill friends wbarr


## Step 3 – Train-Test Split

In [8]:
X = data['cleaned_text']
y = data['Sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
print('Train:', X_train.shape[0], '  Test:', X_test.shape[0])

Train: 40000   Test: 10000


## Step 4 – TF-IDF Vectorisation

Converts each cleaned tweet into a row of numbers. Words that are common across all tweets score low; unique words score high.

In [9]:
tfidf = TfidfVectorizer(max_features=10000)

X_train_tfidf = tfidf.fit_transform(X_train)   # fit on train, transform
X_test_tfidf  = tfidf.transform(X_test)         # only transform (no fitting!)

print('TF-IDF train shape:', X_train_tfidf.shape)
print('Each tweet → row of', X_train_tfidf.shape[1], 'numbers')

TF-IDF train shape: (40000, 10000)
Each tweet → row of 10000 numbers


## Step 5 – Model Training & Evaluation

In [10]:
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)

print('===== Classification Report =====')
print(classification_report(y_test, y_pred,
                            target_names=['Negative (0)', 'Positive (1)']))

===== Classification Report =====
              precision    recall  f1-score   support

Negative (0)       0.88      0.95      0.91      6724
Positive (1)       0.88      0.73      0.80      3276

    accuracy                           0.88     10000
   macro avg       0.88      0.84      0.86     10000
weighted avg       0.88      0.88      0.88     10000



## Bonus – Try Your Own Tweet

In [11]:
def predict_sentiment(tweet):
    """Takes a raw tweet string, cleans it, and predicts sentiment."""
    cleaned    = text_cleaning_pipeline(tweet)
    vectorized = tfidf.transform([cleaned])
    prediction = model.predict(vectorized)[0]
    label      = 'Positive 😊' if prediction == 1 else 'Negative 😠'
    print(f'Tweet     : {tweet}')
    print(f'Cleaned   : {cleaned}')
    print(f'Sentiment : {label}\n')

predict_sentiment('Trump did a great job! The economy is doing amazing!')
predict_sentiment('Trump is ruining everything. This is a disaster!')

Tweet     : Trump did a great job! The economy is doing amazing!
Cleaned   : trump great job economy amaze
Sentiment : Positive 😊

Tweet     : Trump is ruining everything. This is a disaster!
Cleaned   : trump ruin everything disaster
Sentiment : Negative 😠

